# 개별종목 조합G — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합G 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합G의 피처 값만 지정합니다.
import json

COMBINATION = 'G'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_20',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합G 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_20', 'turnover_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4849,0.5012,-0.0163,0.3532,0.3738,0.0873,0.3785,0.1654,0.2743
1,2,balanced,980,20150123,20150421,0.3781,0.3978,-0.0197,0.3439,0.3568,0.0440,0.3669,0.2655,0.3219
2,3,balanced,1210,20151228,20160328,0.3631,0.3762,-0.0130,0.3532,0.3550,0.0372,0.3687,0.2726,0.3242
3,4,balanced,1439,20161202,20170228,0.4649,0.4617,0.0032,0.3959,0.4042,0.1244,0.4219,0.1756,0.2892
4,5,balanced,1669,20171113,20180207,0.3982,0.3901,0.0081,0.3769,0.3815,0.0776,0.3986,0.3072,0.3563
5,6,balanced,1899,20181024,20190118,0.4038,0.3725,0.0313,0.4002,0.4009,0.1045,0.4250,0.3407,0.3793
6,7,balanced,2129,20190930,20191224,0.4508,0.4781,-0.0273,0.3782,0.3847,0.0941,0.4133,0.2442,0.3349
7,8,balanced,2359,20200902,20201130,0.3850,0.3476,0.0374,0.3819,0.3940,0.0900,0.3958,0.5149,0.4191
8,9,balanced,2589,20210806,20211105,0.3626,0.3914,-0.0288,0.3590,0.3729,0.0546,0.3727,0.2902,0.3337
9,10,balanced,2818,20220714,20221012,0.3810,0.3454,0.0355,0.3781,0.3897,0.0829,0.3861,0.2612,0.3297


,OOS 폴드 평균
accuracy,0.4038
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0070
macro_f1,0.3740
balanced_accuracy,0.3818
mcc,0.0794
pr_auc_macro_ovr,0.3934
down_recall,0.3020
core_harmonic_mean,0.3449


재실행 명령: python scripts/run_stock_model_experiment.py
